# IAC Training on TA-RWARE — Google Colab (T4 GPU)

**Project:** Scalable MARL for Warehouse Logistics (Krnjaic et al., 2023 — arXiv 2212.11498v3)  
**Algorithm:** IAC — Independent Actor-Critic (paper Table I, GTP section)  
**Environment:** Small GTP — `tarware-medium-8agvs-4pickers-partialobs-v1`  
**Framework:** [EPyMARL v2](https://github.com/uoe-agents/epymarl) (paper authors' official training framework)

> **Before running:** Go to `Runtime > Change runtime type > T4 GPU`

---

## What this notebook does

Our previous experiments (Exp 1–4) used a rule-based heuristic (CTA). This notebook trains a real
neuronal network using Reinforcement Learning — specifically **IAC**, which is the simplest
algorithm from the paper.

| Method | Small GTP Pick Rate |
|--------|--------------------|
| Random baseline (our Exp 2) | 9.63 ± 0.12 |
| CTA heuristic (our Exp 1) | 52.16 ± 0.54 |
| **IAC neural net (this notebook)** | **~40–65 (3000 ep training)** |
| IAC — paper full training (10k ep) | 65.2 ± 0.5 |
| HIAC — paper best (10k ep) | 66.7 ± 0.3 |

Pick rate = order-lines delivered per hour. Higher is better.

---

## IAC Algorithm (Independent Actor-Critic)

Each agent learns its own **Actor** (policy network) and **Critic** (value network):

```
Observation  ─►  [FC 64] ─► [FC 64]  ─►  Actor head  ─►  Action probabilities
                                      ─►  Critic head ─►  Value estimate
```

- **12 independent agents** (8 AGVs + 4 pickers), each with their own network weights
- Training uses **Generalized Advantage Estimation (GAE)** with γ = 0.99
- No weight sharing between agents (unlike SNAC) — each agent learns independently
- This matches the paper's IAC configuration exactly

## Step 0: Mount Google Drive

We save training results to Drive so they persist after the Colab session ends.
(Training takes 1–2 hours — Drive prevents losing results if the session disconnects.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/tarware_iac_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

## Step 1: Install EPyMARL and TA-RWARE

We install two repos:
- **TA-RWARE** — the warehouse simulation environment (same code as our experiments)
- **EPyMARL** — the official MARL training framework used by the paper authors

In [ ]:
import subprocess, sys

def run(cmd, **kwargs):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print(f'STDERR: {result.stderr[-2000:]}')
    else:
        print(result.stdout[-500:] if result.stdout else 'OK')
    return result.returncode

print('=== Installing TA-RWARE ===')
run('git clone --quiet https://github.com/uoe-agents/task-assignment-robotic-warehouse tarware_repo')
run('pip install -q -e tarware_repo/')

print('\n=== Installing EPyMARL ===')
run('git clone --quiet https://github.com/uoe-agents/epymarl')
run('pip install -q -r epymarl/requirements.txt')

print('\n=== Installing extra dependencies ===')
run('pip install -q pyastar2d sacred pymongo')

print('\nAll installations complete!')

## Step 2: Verify GPU and Environment

Check that:
1. T4 GPU is detected by PyTorch
2. The TA-RWARE environment loads correctly
3. Observation and action spaces look right for 12 agents

In [ ]:
import torch
print('=== GPU Check ===')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')
    print('Go to Runtime > Change runtime type > T4 GPU')

print('\n=== Environment Check ===')
import gymnasium as gym
import tarware  # registers tarware envs with gymnasium

ENV_ID = 'tarware-medium-8agvs-4pickers-partialobs-v1'
env = gym.make(ENV_ID)
obs, info = env.reset(seed=42)

print(f'Environment: {ENV_ID}')
print(f'Number of agents: {env.unwrapped.n_agents}')
print(f'  - AGVs:    {env.unwrapped.num_agvs}')
print(f'  - Pickers: {env.unwrapped.num_pickers}')
print(f'Max steps per episode: {env.unwrapped.max_steps}')

if hasattr(env.observation_space, 'spaces'):
    print(f'Observation space: Tuple of {len(env.observation_space.spaces)} agents')
    for i, sp in enumerate(env.observation_space.spaces[:3]):
        print(f'  Agent {i}: obs shape = {sp.shape}')
    if len(env.observation_space.spaces) > 3:
        print(f'  ... ({len(env.observation_space.spaces)} total)')
else:
    print(f'Observation space: {env.observation_space}')

if hasattr(env.action_space, 'spaces'):
    print(f'Action space: Tuple of {len(env.action_space.spaces)} agents')
    for i, sp in enumerate(env.action_space.spaces[:3]):
        print(f'  Agent {i}: {sp.n} actions')
env.close()
print('\nEnvironment check PASSED!')

## Step 3: Configure IAC Training

Key hyperparameters matching the paper's setup:
- `t_max = 1,500,000` timesteps = 3,000 episodes × 500 steps/episode
- `lr = 0.0005` (learning rate for both actor and critic)
- `gamma = 0.99` (discount factor, as in paper)
- `hidden_dim = 64` (2 FC layers × 64 units, as in paper)

In [ ]:
import os

# Training configuration
ENV_ID       = 'tarware-medium-8agvs-4pickers-partialobs-v1'
T_MAX        = 1_500_000   # 3000 episodes * 500 steps
SEED         = 42
EPYMARL_DIR  = '/content/epymarl'

# EPyMARL Sacred overrides (key=value format)
CONFIG_OVERRIDES = [
    f'env_args.key={ENV_ID}',
    'env_args.time_limit=500',
    'env_args.common_reward=False',    # individual reward per agent
    f't_max={T_MAX}',
    f'seed={SEED}',
    'save_model=True',
    'save_model_interval=500000',
    f'results_save_dir={RESULTS_DIR}',
    'use_tensorboard=False',
]

print('Training Configuration:')
print(f'  Environment  : {ENV_ID}')
print(f'  Algorithm    : IAC (Independent Actor-Critic)')
print(f'  Episodes     : ~{T_MAX // 500:,} (t_max={T_MAX:,} steps)')
print(f'  Seed         : {SEED}')
print(f'  Results dir  : {RESULTS_DIR}')
print(f'\nExpected runtime: 1–2 hours on T4 GPU')

## Step 4: Run IAC Training

This cell launches EPyMARL's training loop via subprocess and streams the output in real-time.

**What to expect:**
- EPyMARL prints progress every few hundred timesteps
- Episode returns start low (random policy) and should increase over time
- Training takes ~1–2 hours on T4 GPU
- Results auto-save to Google Drive

> If the cell is interrupted, the results saved so far are still in Drive.

In [ ]:
import subprocess, sys, time

cmd = [
    sys.executable, 'src/main.py',
    '--config=iac',
    '--env-config=gymma',
    'with',
] + CONFIG_OVERRIDES

print(f'Command: python src/main.py --config=iac --env-config=gymma with [overrides]')
print(f'Full overrides: {CONFIG_OVERRIDES}')
print('=' * 70)
print('Starting training... (output streamed below)')
print('=' * 70)

start_time = time.time()

try:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        cwd=EPYMARL_DIR,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    print('\nTraining interrupted by user.')

elapsed = time.time() - start_time
print('\n' + '=' * 70)
print(f'Exit code  : {proc.returncode}')
print(f'Total time : {elapsed/3600:.2f} hours ({elapsed:.0f} seconds)')
if proc.returncode == 0:
    print('Training completed successfully!')
else:
    print('Training ended with errors. Check output above.')

## Step 5: Load Training Results

EPyMARL uses [Sacred](https://sacred.readthedocs.io) to save experiment results.
Each run creates a numbered directory with `metrics.json` containing all logged values.

In [ ]:
import json, glob, os
import numpy as np

# Find the most recent Sacred run directory
run_dirs = sorted([
    d for d in glob.glob(os.path.join(RESULTS_DIR, '*/')) if os.path.isdir(d)
])

if not run_dirs:
    print('No results found in:', RESULTS_DIR)
    print('Make sure training completed (Step 4) before running this cell.')
else:
    LATEST_RUN = run_dirs[-1]
    print(f'Found {len(run_dirs)} run(s). Loading: {LATEST_RUN}')

    with open(os.path.join(LATEST_RUN, 'metrics.json')) as f:
        metrics = json.load(f)

    print(f'\nAvailable metrics: {list(metrics.keys())}')

    # Show a sample of training progress
    if 'return_mean' in metrics:
        data = metrics['return_mean']['values']
        n = len(data)
        print(f'\nReturn_mean: {n} data points')
        if n > 0:
            print(f'  First: step={data[0]["steps"]}, return={data[0]["value"]:.3f}')
            print(f'  Mid:   step={data[n//2]["steps"]}, return={data[n//2]["value"]:.3f}')
            print(f'  Last:  step={data[-1]["steps"]}, return={data[-1]["value"]:.3f}')

    # Load run config for reference
    config_path = os.path.join(LATEST_RUN, 'config.json')
    if os.path.exists(config_path):
        with open(config_path) as f:
            run_config = json.load(f)
        print(f'\nRun config t_max: {run_config.get("t_max", "N/A")}')
        print(f'Run config seed:  {run_config.get("seed", "N/A")}')
    print('\nResults loaded successfully!')

## Step 6: Plot Learning Curves

We plot two views:
1. **Raw episode return** — what EPyMARL logs directly
2. **Approximate pick rate** — converted using the paper's formula

**Conversion formula:**  
`pick_rate ≈ return_sum * 3600 / (5 * 500)`  
where `return_sum = return_mean_per_agent × 12 agents`

Reference lines show how our IAC compares to:
- Our CTA heuristic (Exp 1): **52.16**
- Paper's CTA: **52.7**  
- Paper's IAC (full 10k ep): **65.2**
- Paper's HIAC (best): **66.7**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

N_AGENTS = 12   # 8 AGVs + 4 pickers
MAX_STEPS = 500  # steps per episode

# Smooth a noisy signal with a rolling window
def smooth(values, window=20):
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window)/window, mode='valid')

if 'return_mean' not in metrics:
    print('return_mean not found in metrics. Available:', list(metrics.keys()))
else:
    data = metrics['return_mean']['values']
    steps = np.array([v['steps'] for v in data])
    returns_raw = np.array([v['value'] for v in data])

    # Convert to approximate pick rate
    # return_mean is per-agent; multiply by N_AGENTS to get total deliveries
    pick_rate_approx = returns_raw * N_AGENTS * 3600 / (5 * MAX_STEPS)

    steps_sm = steps[len(steps)-len(smooth(returns_raw)):]
    returns_sm = smooth(returns_raw)
    pick_sm = smooth(pick_rate_approx)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('IAC Training — Small GTP (tarware-medium-8agvs-4pickers)', fontsize=13)

    # Left: raw return
    ax = axes[0]
    ax.plot(steps, returns_raw, alpha=0.25, color='steelblue')
    ax.plot(steps_sm, returns_sm, color='steelblue', linewidth=2, label='IAC return (smoothed)')
    ax.set_xlabel('Training Timesteps')
    ax.set_ylabel('Mean Episode Return (per agent)')
    ax.set_title('Raw Episode Return')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

    # Right: approximate pick rate with reference lines
    ax = axes[1]
    ax.plot(steps, pick_rate_approx, alpha=0.25, color='steelblue')
    ax.plot(steps_sm, pick_sm, color='steelblue', linewidth=2, label='IAC (ours)')
    ax.axhline(y=52.16, color='darkorange', linestyle='--', linewidth=2,
               label='CTA ours (52.16)')
    ax.axhline(y=52.7,  color='gray',       linestyle=':',  linewidth=1.5,
               label='CTA paper (52.7)')
    ax.axhline(y=65.2,  color='green',      linestyle=':',  linewidth=1.5,
               label='IAC paper 10k ep (65.2)')
    ax.axhline(y=66.7,  color='red',        linestyle=':',  linewidth=1.5,
               label='HIAC paper 10k ep (66.7)')
    ax.set_xlabel('Training Timesteps')
    ax.set_ylabel('Approx. Pick Rate (order-lines / hr)')
    ax.set_title('Approximate Pick Rate vs Baselines')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

    plt.tight_layout()
    plot_path = os.path.join(RESULTS_DIR, 'iac_training_curve.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f'Plot saved to: {plot_path}')
    plt.show()

    final_return = float(returns_sm[-1])
    final_pick_rate = float(pick_sm[-1])
    print(f'\nFinal smoothed return   : {final_return:.3f}')
    print(f'Approx. final pick rate : {final_pick_rate:.2f}')

## Step 7: Performance Summary Table

Full comparison from Random baseline → CTA heuristic → IAC (ours) → paper targets.

In [ ]:
import json, glob, os
import numpy as np

# Reference values from our experiments and the paper
RESULTS = [
    ('Random (our Exp 2)',          9.63,  0.12, 'ours'),
    ('CTA heuristic (our Exp 1)',   52.16, 0.54, 'ours'),
    ('CTA (paper Table I)',         52.7,  0.9,  'paper'),
    ('IAC (this notebook, 3k ep)',  None,  None, 'ours'),
    ('IAC (paper, 10k ep)',         65.2,  0.5,  'paper'),
    ('HIAC (paper, 10k ep)',        66.7,  0.3,  'paper'),
]

# Try to fill in our IAC result
try:
    iac_pick_rate = final_pick_rate
except NameError:
    iac_pick_rate = None

for i, (name, val, ci, src) in enumerate(RESULTS):
    if name.startswith('IAC (this notebook'):
        RESULTS[i] = (name, iac_pick_rate, None, src)

print('=' * 68)
print(f'PERFORMANCE COMPARISON — Small GTP (tarware-medium-8agvs-4pickers)')
print('=' * 68)
print(f'{"Method":<35} {"Pick Rate":>14} {"Source":>8}')
print('-' * 68)
for name, val, ci, src in RESULTS:
    if val is None:
        val_str = '[not available]'
    elif ci is not None:
        val_str = f'{val:.2f} +/- {ci:.2f}'
    else:
        val_str = f'~{val:.1f} (approx)'
    print(f'{name:<35} {val_str:>14} {src:>8}')
print('=' * 68)
print()
print('Note: IAC pick rate is approximate (estimated from episode returns).')
print('      The paper trained for 10,000 episodes; we trained for ~3,000.')
print()
print('Key insight: IAC neural network trains FROM SCRATCH and should')
print('exceed the CTA rule-based heuristic after enough training.')
print('The gap between IAC(paper) and HIAC(paper) motivates hierarchical MARL.')

# Save summary to Drive
summary_data = {
    'method': [r[0] for r in RESULTS],
    'pick_rate': [r[1] for r in RESULTS],
    'ci': [r[2] for r in RESULTS],
    'source': [r[3] for r in RESULTS],
}
with open(os.path.join(RESULTS_DIR, 'iac_summary.json'), 'w') as f:
    json.dump(summary_data, f, indent=2)
print(f'\nSummary saved to: {os.path.join(RESULTS_DIR, "iac_summary.json")}')

---

## Notes for Report and Presentation

### What to report

1. **Algorithm**: IAC (Independent Actor-Critic) — each of 12 agents (8 AGVs + 4 pickers) maintains its own 2-layer neural network (64 units, ReLU) for both actor (policy) and critic (value estimation). Training uses GAE with γ=0.99.

2. **Framework**: EPyMARL v2 — the paper authors' own training framework, ensuring methodology is faithful to the original paper.

3. **Training setup**: 3,000 episodes (~1.5M timesteps) on Google Colab T4 GPU (~1–2 hours). The paper trained for 10,000 episodes with 5 random seeds.

4. **Result interpretation**:
   - If our IAC exceeds CTA (52.16): neural learning works, RL adds value beyond rule-based methods
   - If not yet converged: the *learning curve* itself is the deliverable — it shows the ML training is happening and heading in the right direction
   - Gap between ours (3k ep) and paper (10k ep) motivates the value of more compute

### Why IAC and not HIAC?

IAC is the simplest algorithm in the paper. HIAC adds a hierarchical manager network on top, making it significantly more complex. For a student project:
- IAC demonstrates the core RL concept clearly
- IAC results are directly comparable to paper Table I
- HIAC is the 'future work' — requires weeks more training

### Troubleshooting

- **`ModuleNotFoundError: tarware`**: Re-run the install cell
- **`No GPU detected`**: Runtime > Change runtime type > T4 GPU, then Restart runtime
- **Training very slow**: Verify GPU is active (should see 200+ FPS in EPyMARL output)
- **Sacred errors**: EPyMARL may need `pip install pymongo` for MongoDB observer
- **`env_args.key` error**: Try quoting the env ID: `with "env_args.key=tarware-medium-..."`